In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Kurumsal Bilgi Yönetimi

Amaç: Bir çalışanın, şirketin iç yönetmelikleri (izin, masraf vb.) hakkında sorduğu bir soruya, ilgili dokümanlardan cevap bulmak.

In [ ]:
pip install openai


In [ ]:
import os
import openai

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
openai.api_key  = user_secrets.get_secret("OPENAI")

def ask_chatgpt(prompt):
    response = openai.chat.completions.create(
        model="gpt-4",  # veya "gpt-3.5-turbo"
        messages=[
            {"role": "system", "content": "Sen faydalı bir yardımcı botsun."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=500,
    )
    
    return response.choices[0].message.content

# Örnek soru
question = "Kuantum bilgisayarlar klasik bilgisayarlardan nasıl farklı çalışır?"
answer = ask_chatgpt(question)

print("Soru:", question)
print("Cevap:", answer)


In [ ]:
# Gerekli kütüphaneler: pip install sentence-transformers scikit-learn numpy
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- BİLGİ KAYNAĞI ---
KNOWLEDGE_BASE = """
İnsan Kaynakları Politikası:
Şirketimiz, çalışanlarının profesyonel gelişimini destekleyen, adil ve kapsayıcı bir çalışma ortamı sunmayı amaçlayan bir insan kaynakları politikasına sahiptir. İşe alım süreçlerinde eşit fırsat ilkesi benimsenmekte, tüm değerlendirmeler objektif kriterler çerçevesinde yapılmaktadır. Terfi ve performans değerlendirmelerinde şeffaflık sağlanmakta, çalışanlara düzenli geri bildirim verilerek gelişim alanları belirlenmektedir. Ayrıca şirket, çeşitlilik ve kapsayıcılığı stratejik öncelik olarak görmekte, tüm çalışanların potansiyellerini tam anlamıyla ortaya koyabilecekleri güvenli bir iş ortamı yaratmaktadır.

Global Çalışan El Kitabı:
Bu doküman, şirketin farklı ülkelerdeki ofislerinde görev yapan tüm çalışanlar için ortak değerleri, temel kuralları ve prosedürleri tanımlar. İş etiği, gizlilik, sosyal medya kullanımı, iş güvenliği ve vardiya sistemleri gibi başlıklar global ölçekte standartlaştırılmıştır. Bununla birlikte, her ülkenin yerel mevzuatına uyumlu ek yönergeler de sunulmuştur. El kitabı, tüm çalışanlar için bir referans kaynağı niteliği taşır ve yeni çalışanların oryantasyon süreçlerinde temel rehber olarak kullanılmaktadır.

Ücretlendirme ve Yan Haklar Politikası:
Şirketimiz, uluslararası pazardaki rekabetçiliğini koruyabilmek amacıyla çalışanlarına adil ve piyasa koşullarına uygun bir ücretlendirme sistemi sunar. Sabit maaşların yanında, performansa dayalı bonus ve yıllık prim sistemleri uygulanmaktadır. Ayrıca, sağlık sigortası, emeklilik planları, hisse opsiyonu ve esnek yan haklar gibi ek avantajlar da çalışanların motivasyonunu artırmayı hedeflemektedir. Ücretlendirme sistemi yılda en az bir kez gözden geçirilerek piyasa analizleri ve çalışan geri bildirimleri doğrultusunda güncellenmektedir.

Uzaktan Çalışma ve Hibrit Model Prosedürü:
Şirketin uzaktan ve hibrit çalışma politikası, hem verimliliği hem de çalışanların iş-yaşam dengesini destekleyecek şekilde yapılandırılmıştır. Uzaktan çalışmanın kapsamı, iletişim kanalları, güvenlik önlemleri ve performans takibi gibi konular net bir şekilde tanımlanmıştır. Tüm çalışanların bilgi güvenliği protokollerine uyması zorunludur ve kullanılan dijital platformların erişim yetkileri düzenli olarak kontrol edilmektedir. Hibrit modelde ise belirli günlerde ofiste bulunma zorunluluğu, ekip uyumu ve iletişimi güçlendirmek için planlı şekilde organize edilir.

Transfer Fiyatlandırması Raporları:
Çok uluslu yapımız gereği, grup şirketleri arasında gerçekleştirilen mal ve hizmet transferleri için uluslararası transfer fiyatlandırması ilkeleri uygulanır. OECD’nin belirlediği “emsallere uygunluk” prensibi temel alınarak hazırlanan bu raporlar, vergi otoriteleri ile tam uyumluluk sağlamak amacıyla düzenlenir. Her yıl güncellenen transfer fiyatlandırması raporları, hem mali şeffaflığı artırmak hem de olası cezai yaptırımların önüne geçmek için kritik bir araçtır.

IFRS/UFRS Finansal Raporlar:
Şirketimiz, finansal sonuçlarını Uluslararası Finansal Raporlama Standartları (IFRS/UFRS) doğrultusunda hazırlar ve yayınlar. Bu raporlar, yatırımcıların, hissedarların ve regülasyon kurumlarının şirketin mali durumu hakkında objektif ve şeffaf bilgi edinmesini sağlar. Finansal tablolar, bağımsız denetim firmaları tarafından yılda bir kez incelenir ve doğrulanır. Bu raporlar; bilanço, gelir tablosu, nakit akış tablosu ve özkaynak değişim tablosu gibi temel unsurları içerir.

Bütçe Planlama Kılavuzu:
Yıllık ve çeyrek dönemli bütçe planlama süreçlerinin standartlaştırılması için hazırlanan bu kılavuz, her departmanın mali hedeflerini ve kaynak kullanım stratejilerini belirlemesine yardımcı olur. Gelir ve gider projeksiyonları, operasyonel maliyetlerin optimizasyonu ve sermaye harcamalarının planlanması gibi unsurlar, bu kılavuz çerçevesinde koordine edilir. Tüm bütçe revizyonları, yönetim kurulunun onayı ile yürürlüğe girer.

Vergi Uyum Raporları:
Şirketin faaliyet gösterdiği tüm ülkelerdeki yerel ve uluslararası vergi mevzuatına uyum sağlamak amacıyla yıllık vergi raporları hazırlanır. BEPS (Base Erosion and Profit Shifting), FATCA ve CRS gibi global düzenlemeler doğrultusunda düzenlenen bu raporlar, hem şeffaflık hem de vergi yükümlülüklerinin eksiksiz yerine getirilmesi açısından kritik öneme sahiptir.

İç Kontrol Matrisleri (RACI):
Finansal süreçlerde yetki ve sorumlulukların netleştirilmesi amacıyla RACI (Responsible, Accountable, Consulted, Informed) metodolojisi ile iç kontrol matrisleri oluşturulmuştur. Bu matrisler, muhasebe, ödeme, bütçe onayı ve harcama politikalarının her bir adımında kimin sorumlu, kimin onaylayıcı veya bilgilendirici rol üstlendiğini açıkça ortaya koyar. Böylece hata payı en aza indirilir ve kurumsal risk yönetimi güçlendirilir.

Çalışan Geri Bildirim ve Şikayet Süreci:
Şirket, çalışanlarının sesini duyurabilmesi için anonim geri bildirim kanalları ve etik hatlar oluşturmuştur. Çalışanlar iş ortamına, yönetim süreçlerine veya çalışma arkadaşlarına dair geri bildirimlerini belirlenen dijital platformlar üzerinden iletebilir. Gelen tüm geri bildirimler gizlilik esasına uygun şekilde İnsan Kaynakları departmanı tarafından değerlendirilir ve gerekli durumlarda bağımsız denetim mekanizmaları devreye sokulur. Şikayet süreçleri, çözüm odaklı ve adil bir yaklaşım çerçevesinde yürütülür.
Madde 12: Yıllık İzin Politikası. Şirketimizde bir tam yılı dolduran her çalışan 14 iş günü ücretli yıllık izin hakkına sahiptir. Beş yıl ve üzeri kıdeme sahip çalışanlar için bu süre 20 iş günüdür. Kullanılmayan izinler bir sonraki yıla devretmez, ancak şirket yönetimi onayı ile en fazla 5 günü devredebilir.

Madde 13: Masraf Beyanı Politikası. Çalışanlar, şirket için yaptıkları seyahat, konaklama ve yemek masraflarını ayın son Cuma gününe kadar insan kaynakları departmanına bildirmelidir. Faturaların asıllarının beyana eklenmesi zorunludur. Onaylanan masraflar bir sonraki ayın maaşı ile birlikte ödenir.

Madde 14: Uzaktan Çalışma Politikası. Haftada iki gün uzaktan çalışma hakkı, departman yöneticisinin onayı ile tüm çalışanlara tanınmıştır. Uzaktan çalışılan günlerde çalışanların standart mesai saatleri içinde ulaşılabilir olması beklenir.
"""

# --- MODÜLER FONKSİYONLAR ---
def run_semantic_chunking(text: str, model, threshold: float = 0.45) -> tuple[list[str], list[float]]:
    """
    Metni cümlelere ayırır ve anlamsal benzerlik eşiğine göre chunk'lar oluşturur.
    Ara çıktı olarak benzerlik skorlarını da döndürür.
    """
    print(f"Parametreler: similarity_breakpoint_threshold={threshold}")
    # Metni cümlelere böl (basit bir regex ile)
    sentences = re.split(r'(?<=[.!?])\s+', text.replace("\n", " ").strip())
    
    # Her cümlenin embedding'ini hesapla
    embeddings = model.encode(sentences)
    
    # Ardışık cümleler arası kosinüs benzerliğini hesapla
    similarities = []
    for i in range(len(sentences) - 1):
        emb1 = embeddings[i].reshape(1, -1)
        emb2 = embeddings[i+1].reshape(1, -1)
        sim = cosine_similarity(emb1, emb2)[0][0]
        similarities.append(sim)
        
    # Eşik değerinin altına düşen yerleri kırılma noktası olarak belirle
    breakpoint_indices = [i + 1 for i, sim in enumerate(similarities) if sim < threshold]
    
    # Kırılma noktalarına göre chunk'ları oluştur
    chunks = []
    start_index = 0
    for break_index in breakpoint_indices:
        chunk = " ".join(sentences[start_index:break_index])
        chunks.append(chunk)
        start_index = break_index
    chunks.append(" ".join(sentences[start_index:])) # Son chunk'ı ekle
    
    return chunks, similarities
    
def chunk_text(text: str) -> list[str]:
    """Metni paragraflarına göre böler."""
    return [p.strip() for p in text.split('\n\n') if p.strip()]

def get_embeddings(chunks, model):
    return model.encode(chunks)

def find_most_similar_chunks(query, embeddings, chunks, model, top_k=2):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, embeddings)[0]

    top_k_indices = np.argsort(similarities)[-top_k:][::-1]
    
    
    l= [chunks[i] for i in top_k_indices]

    return l

def generate_answer(query, context):
    prompt = f"Bağlam: {context}\n\nSoru: {query}\n\nCevap:"
    print("--- LLM'e Gönderilen Nihai Prompt ---\n", prompt)
    mock_response = f"Şirket politikasına göre, '{query}' sorunuzun cevabı şudur: {context}"
    return mock_response

def display_results(strategy_name: str, chunks: list, extra_info=None):
    """
    Her stratejinin sonucunu standart bir formatta ekrana basar.
    """
    print("\n" + "="*50)
    print(f"STRATEJİ: {strategy_name}")
    print("="*50)
    
    if extra_info:
        if "similarities" in extra_info:
            print("--- Ara Çıktı: Cümleler Arası Benzerlik Skorları ---")
            for i, sim in enumerate(extra_info["similarities"]):
                print(f"  Cümle {i+1} <-> Cümle {i+2} Benzerlik: {sim:.2f}")
            print("-" * 20)

    print("\n--- Oluşturulan Chunk'lar ---")
    for i, chunk in enumerate(chunks):
        print(f"CHUNK {i+1}:")
        print(f'"{chunk.strip()}"')
        print(f"(Uzunluk: {len(chunk)} karakter)\n")
    print("\n")

# --- ANA PROGRAM AKIŞI ---
if __name__ == "__main__":
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    user_query = "çalışanlara sunulan ücretlendirme sistemi nasıldır"
    
    print("--- Kurumsal Bilgi Yönetimi RAG Demosu ---")
    print(f"Kullanıcı Sorusu: '{user_query}'\n")

    # 1. Parçalama
    semantic_chunks, similarities = run_semantic_chunking(KNOWLEDGE_BASE, embedding_model)
    display_results("Semantic Chunking", semantic_chunks, extra_info={"similarities": similarities})
    print(f"--- 1. Parçalama Sonucu: {len(chunks)} adet chunk (madde) bulundu. ---\n")
    
    # 2. Gömme
    document_embeddings = get_embeddings(semantic_chunks, embedding_model)
    
    
    # 3. Arama
    context_chunks = find_most_similar_chunks(user_query, document_embeddings, semantic_chunks, embedding_model)
    
    context_for_llm = "\n".join(context_chunks)
    print("--- 2. Arama Sonucu: En Alakalı Madde Bulundu ---\n", context_for_llm, "\n")

    # 4. Üretim 1. yöntem
    query = generate_answer(user_query, context_for_llm)
    print("query : llm'e gitmeden önce: ", query)
    print("\n--- 3. Nihai Cevap Üretildi ---")
    llm_cevabi =  ask_chatgpt(query)
    print('ilk Cevap #1. Yöntem: ', llm_cevabi)

    #5. alternatif llm yöntemi
    llm_cevabi = []
    yeni_sorgu = ''
    for chunk in context_chunks:
        cevap = ask_chatgpt("şu bilgileri alıp en altta sorduğum soruyu cevapla: "+ chunk+ '\n'+ "şimdi sorumu cevapla : "+ query)
        llm_cevabi+= [cevap]
        yeni_sorgu += cevap
    
    answer = ask_chatgpt(yeni_sorgu)
    print('Yeni Cevap #2. Yöntem', answer)


In [ ]:
llm_cevabi